In [ ]:
!pip install pandas scikit-learn joblib

In [ ]:
!pip install seaborn

In [ ]:
!pip install streamlit pyngrok

In [ ]:
import pandas as pd
import random

categories = {
    "network": [
        "Cannot connect to WiFi",
        "Internet is very slow",
        "Network cable unplugged",
        "VPN not connecting",
        "DNS server not responding",
        "Frequent network disconnection",
        "IP address conflict detected",
        "Router keeps restarting"
    ],
    "hardware": [
        "Laptop is overheating",
        "Keyboard not working",
        "Mouse is not detected",
        "Screen flickering",
        "Battery not charging",
        "Hard drive failure",
        "Computer not turning on",
        "USB port not working"
    ],
    "software": [
        "MS Word keeps crashing",
        "Application not responding",
        "Unable to install software",
        "System update failed",
        "Blue screen error",
        "Antivirus not updating",
        "Software license expired",
        "File cannot be opened"
    ],
    "account": [
        "Forgot my email password",
        "Account locked after login attempts",
        "Unable to access shared drive",
        "Permission denied error",
        "Need password reset",
        "Two-factor authentication not working",
        "Cannot login to system",
        "Email not syncing"
    ],
    "printer": [
        "Printer not responding",
        "Paper jam error",
        "Printer offline",
        "Cannot print document",
        "Low ink warning",
        "Printer driver missing",
        "Printer printing blank pages",
        "Printer not detected"
    ]
}

actions = {
    "network": "Restart router and check cables",
    "hardware": "Check physical connections or repair hardware",
    "software": "Reinstall or update the software",
    "account": "Reset password or verify account settings",
    "printer": "Restart printer and check paper/ink"
}

data = []

for category, texts in categories.items():
    for _ in range(45):  # 45 x 5 categories = 225 records
        text = random.choice(texts)
        data.append({
            "text": text,
            "category": category,
            "action": actions[category]
        })

df = pd.DataFrame(data)
df.to_csv("helpdesk_tickets.csv", index=False)

print("Dataset created successfully!")
print("Total records:", len(df))
df.head()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Load dataset
data = pd.read_csv("helpdesk_tickets.csv")

X = data["text"]
y = data["category"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert text to numbers
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),   # unigram + bigram
    max_features=1000
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("Training Accuracy:", model.score(X_train_vec, y_train))
print("Testing Accuracy:", model.score(X_test_vec, y_test))

# Evaluate
y_pred = model.predict(X_test_vec)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = model.classes_
scores = model.score(X_test_vec, y_test)

plt.figure()
plt.bar(["Test Accuracy"], [scores])
plt.title("Model Performance")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=model.classes_,
            yticklabels=model.classes_)
plt.title("Confusion Matrix Heatmap")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
while True:
    text = input("Enter IT issue (type exit to stop): ")
    if text.lower() == "exit":
        break

    vec = vectorizer.transform([text])
    prediction = model.predict(vec)[0]

    print("\n--- Helpdesk Advisory Result ---")
    print("Predicted Category:", prediction)

    if prediction == "network":
        print("Suggested Action: Restart router and check cables.")
    elif prediction == "hardware":
        print("Suggested Action: Inspect hardware components.")
    elif prediction == "software":
        print("Suggested Action: Reinstall or update the application.")
    elif prediction == "account":
        print("Suggested Action: Reset or verify account credentials.")
    elif prediction == "printer":
        print("Suggested Action: Check printer connection and ink levels.")
    print("--------------------------------\n")



*   List item
*   List item



In [ ]:
import joblib

joblib.dump(model, "helpdesk_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model files saved successfully.")

In [ ]:
%%writefile app.py
import streamlit as st
import joblib

model = joblib.load("helpdesk_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

st.title("IT Helpdesk Advisory System")

user_input = st.text_input("Describe your issue:")

if st.button("Predict"):
    vec = vectorizer.transform([user_input])
    prediction = model.predict(vec)[0]
    st.success(f"Predicted Category: {prediction}")

In [ ]:
!ls

In [ ]:
%%writefile app.py